In [1]:
!nvidia-smi

Fri Jun 12 09:03:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!git clone https://github.com/AlexeyAB/darknet
%cd darknet

Cloning into 'darknet'...
remote: Enumerating objects: 15941, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 15941 (delta 13), reused 4 (delta 4), pack-reused 15914 (from 3)
Receiving objects: 100% (15941/15941), 14.52 MiB | 14.12 MiB/s, done.
Resolving deltas: 100% (10719/10719), done.
/content/darknet


In [4]:
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile

In [5]:
!make

mkdir -p ./obj/
mkdir -p backup
mkdir -p results
chmod +x *.sh
g++ -std=c++11 -std=c++11 -Iinclude/ -I3rdparty/stb/include -DOPENCV `pkg-config --cflags opencv4 2> /dev/null || pkg-config --cflags opencv` -DGPU -I/usr/local/cuda/include/ -DCUDNN -Wall -Wfatal-errors -Wno-unused-result -Wno-unknown-pragmas -fPIC -rdynamic -Ofast -DOPENCV -DGPU -DCUDNN -I/usr/local/cudnn/include -c ./src/image_opencv.cpp -o obj/image_opencv.o
Package opencv was not found in the pkg-config search path.
Perhaps you should add the directory containing `opencv.pc'
to the PKG_CONFIG_PATH environment variable
Package 'opencv', required by 'virtual:world', not found
./src/image_opencv.cpp:16:10: fatal error: opencv2/core/version.hpp: No such file or directory
   16 | #include <opencv2/core/version.hpp>
      |          ^~~~~~~~~~~~~~~~~~~~~~~~~~
compilation terminated.
make: *** [Makefile:198: obj/image_opencv.o] Error 1


In [6]:
import os

with open('data/train.txt','w') as f:
    for root, dirs, files in os.walk('/content/drive/MyDrive/yolov4/train/images'):
        for file in files:
            if file.endswith('.jpg'):
                f.write(os.path.join(root,file)+'\n')

In [7]:
!wget https://pjreddie.com/media/files/darknet53.conv.74

--2026-06-12 09:09:32--  https://pjreddie.com/media/files/darknet53.conv.74
Resolving pjreddie.com (pjreddie.com)... 172.67.185.199, 104.21.88.156, 2606:4700:3030::ac43:b9c7, ...
Connecting to pjreddie.com (pjreddie.com)|172.67.185.199|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://data.pjreddie.com/files/darknet53.conv.74 [following]
--2026-06-12 09:09:33--  https://data.pjreddie.com/files/darknet53.conv.74
Resolving data.pjreddie.com (data.pjreddie.com)... 172.67.185.199, 104.21.88.156, 2606:4700:3037::6815:589c, ...
Connecting to data.pjreddie.com (data.pjreddie.com)|172.67.185.199|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 162482580 (155M) [application/octet-stream]
Saving to: ‘darknet53.conv.74’

darknet53.conv.74   100%[===================>] 154.96M   107MB/s    in 1.4s    

2026-06-12 09:09:34 (107 MB/s) - ‘darknet53.conv.74’ saved [162482580/162482580]



In [8]:
!cp cfg/yolov4-custom.cfg cfg/yolov4-obj.cfg

In [9]:
!./darknet detector train \
data/obj.data \
cfg/yolov4-obj.cfg \
darknet53.conv.74 \
-dont_show

/bin/bash: line 1: ./darknet: No such file or directory
